# GDS Assignment 2 
#### Markkus Phamuang Hansen - xfj774

## Part A

In [17]:
import numpy as np

### A.1

In [18]:
data = np.genfromtxt('play_tennis.csv', dtype=str, delimiter=',', skip_header=1)
X = data[:, 0:4]
y = data[:, 4:].ravel()

### A.2

In [19]:
N = len(y)
y_labels = np.unique(y)
values = [np.unique(X[:, i]) for i in range(X.shape[1])]
print(f'Number of rows N: {N} \n'
      f'Unique class labels in y: {y_labels} \n'
      f'Unique values per feature column: {[[str(label) for label in value] for value  in values]}')

Number of rows N: 14 
Unique class labels in y: ['No' 'Yes'] 
Unique values per feature column: [['Overcast', 'Rainy', 'Sunny'], ['Cool', 'Hot', 'Mild'], ['High', 'Normal'], ['Strong', 'Weak']]


## Part B

### B.1

Priors are printed in the cell output below. Class counts can be seen in the numerators, where the class count of 'No' is 5, and the class count of 'Yes' is 9

In [20]:
P = {c: np.sum(y == c)/N for c in y_labels}
for c in y_labels:
      print(f'P({c}) = {np.sum(y == c)}/{N} = {P[c]}')

P(No) = 5/14 = 0.35714285714285715
P(Yes) = 9/14 = 0.6428571428571429


## Part C

In [21]:
x_star = ['Sunny', 'Cool', 'High', 'Strong']

### C.1

In [22]:
def cond_prob(X_i, v, y, c):
    c_mask = (y == c)
    return np.sum((X_i == v) & c_mask)/np.sum(c_mask)

### C.2

The conditional probabilities are expressed as fractions and decimals in the code output below

In [23]:
for c in ['Yes', 'No']:
    print(f'c = {c}: ')
    for i in range(X.shape[1]):
        c_mask = (y == c)
        print(f'    P({x_star[i]}|{c}) = '
              f'{np.sum((X[:, i] == x_star[i]) & c_mask)}/{np.sum(c_mask)} = '
              f'{cond_prob(X[:, i], x_star[i], y, c)}')

c = Yes: 
    P(Sunny|Yes) = 2/9 = 0.2222222222222222
    P(Cool|Yes) = 3/9 = 0.3333333333333333
    P(High|Yes) = 3/9 = 0.3333333333333333
    P(Strong|Yes) = 3/9 = 0.3333333333333333
c = No: 
    P(Sunny|No) = 3/5 = 0.6
    P(Cool|No) = 1/5 = 0.2
    P(High|No) = 4/5 = 0.8
    P(Strong|No) = 3/5 = 0.6


## Part D

### D.1

In [24]:
scores = {
    c: P[c] * np.prod([cond_prob(X[:, i], x_star[i], y, c) for i in range(X.shape[1])])
    for c in y_labels
}
for (c,s) in scores.items():
    print(f'score({c}) = {s}')

score(No) = 0.02057142857142857
score(Yes) = 0.005291005291005291


### D.2
'No' is predicted:

In [25]:
print(
    f'argmax{tuple(round(float(s), 4) for s in scores.values())} '
    f'= {max(scores.keys(), key = lambda c: scores[c])}'
)

argmax(0.0206, 0.0053) = No


## Part E

### E.1

I added the parameter "cond_prob_function" to make my solutions in part F simpler

In [26]:
def fit_naive_bayes(X, y, cond_prob_function=cond_prob):
    N = len(y)
    model = {}
    model['classes'] = np.unique(y)
    model['priors'] = {c: np.sum(y == c)/N for c in model['classes']}
    model['categories'] = [np.unique(X[:, i]) for i in range(len(X[0]))]
    model['cond'] = [
        {
            c: {
                v: cond_prob_function(X[:, i], v, y, c)
                for v in model['categories'][i]
            }
            for c in model['classes']
        }
        for i in range(len(X[0]))
    ]
    return model

### E.2

In [27]:
def predict_one(obs, model):
    n = len(obs)
    return max(
        model['classes'],
        key = lambda c: model['priors'][c] * np.prod(
            [model['cond'][j][c][obs[j]] for j in range(n)]
        )
    )

def predict(X, model):
    return np.array([predict_one(obs, model) for obs in X])

model = fit_naive_bayes(X, y)
print(' ----- TESTING predict(X, model) ----- ')
print(f'Predicted labels for rows in play_tennis.csv using predict(X, model): \n{predict(X, model)}')
print(f'Actual labels for rows in play_tennis.csv: \n{y}')


 ----- TESTING predict(X, model) ----- 
Predicted labels for rows in play_tennis.csv using predict(X, model): 
['No' 'No' 'Yes' 'Yes' 'Yes' 'Yes' 'Yes' 'No' 'Yes' 'Yes' 'Yes' 'Yes'
 'Yes' 'No']
Actual labels for rows in play_tennis.csv: 
['No' 'No' 'Yes' 'Yes' 'Yes' 'No' 'Yes' 'No' 'Yes' 'Yes' 'Yes' 'Yes' 'Yes'
 'No']


### E.3
'No' is still predicted. 

In [28]:
model = fit_naive_bayes(X, y)
print(f'Prediction: {predict_one(x_star, model)}')

Prediction: No


## Part F

### F.1

In [29]:
model = fit_naive_bayes(X, y)
print(f'P(Overcast|No) = {model['cond'][0]['No']['Overcast']}')

P(Overcast|No) = 0.0


### F.2

In [30]:
def laplace_cond_prob(X_i, v, y, c):
    K_i = np.unique(X_i)
    c_mask = (y == c)
    return (np.sum((X_i == v) & c_mask) + 1)/(np.sum(c_mask) + K_i.shape[0])

### F.3

In [31]:
model = fit_naive_bayes(X, y, laplace_cond_prob)
print(f'P(Overcast|No) = {model['cond'][0]['No']['Overcast']}')

P(Overcast|No) = 0.125
